# Knowledge-Graph Anomaly Detection on Google Colab

This notebook runs the [`pipeline.py`](pipeline.py) end-to-end on a **GPU** so
training and evaluation finish in a fraction of the CPU run time.

## Before you start: enable the GPU

Go to **Runtime → Change runtime type → Hardware accelerator** and pick **GPU**
(any of T4/L4/A100 works; the pipeline defaults are tuned for the **A100**).
PyKEEN auto-detects CUDA, so no code changes are needed – the pipeline will use
the GPU automatically once one is attached, enable Ampere/A100 TF32 tensor-core
math, and size its batches for the GPU.

Run the cells below in order.

## 1. Confirm a GPU is attached

In [ ]:
!nvidia-smi

## 2. Clone the repository

Replace the URL if you are working from a fork.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/arslansmajevic/knowledge-graphs-project.git"
REPO_DIR = REPO_URL.rsplit("/", 1)[-1].removesuffix(".git")
REPO_PATH = Path("/content") / REPO_DIR

# Clone only if the repository is not already present
if not REPO_PATH.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_PATH)], check=True)
elif not (REPO_PATH / ".git").exists():
    raise RuntimeError(f"{REPO_PATH} exists but is not a Git repository.")

# Move into the repo and get the newest committed changes
os.chdir(REPO_PATH)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "pull", "--ff-only"], check=True)

# Show latest commits
subprocess.run(["git", "log", "--oneline", "-10"], check=True)

print(f"\nCurrent directory: {os.getcwd()}")

## 3. Install the dependencies

PyKEEN pulls in a CUDA-enabled PyTorch automatically on Colab.

In [ ]:
!pip install -q -r requirements.txt

## 4. Provide the dataset

The raw LANL logs are **not** committed to the repo (they are far too large), so
you need to make them available to Colab. The pipeline expects them in a
`dataset/` directory at the repo root:

```
dataset/
├── auth.txt
├── proc.txt
├── flows.txt
├── dns.txt
└── redteam.txt
```

The easiest option is to keep the files in **Google Drive** and point the repo's
`dataset/` at them. Upload your (optionally gzipped) LANL files to a folder in
Drive once, then run the cell below. It is preset to your dataset folder
(*My Drive > TU Wien > MA 3 > KG VU > dataset*); adjust `DRIVE_DATASET_DIR` if you
move it.

> Colab sessions are ephemeral – anything not in Drive is lost when the runtime
> recycles, so storing the dataset in Drive saves you from re-uploading.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Folder in your Drive that contains auth.txt, proc.txt, flows.txt, dns.txt, redteam.txt
# My Drive > TU Wien > MA 3 > KG VU > dataset
DRIVE_DATASET_DIR = '/content/drive/MyDrive/TU Wien/MA 3/KG VU/dataset'

# Symlink it as ./dataset so the pipeline finds the files unchanged.
if os.path.islink('dataset') or os.path.exists('dataset'):
    print('dataset/ already present:', os.listdir('dataset'))
else:
    os.symlink(DRIVE_DATASET_DIR, 'dataset')
    print('Linked dataset ->', DRIVE_DATASET_DIR)
    print(os.listdir('dataset'))

<details>
<summary>Alternative: upload the files directly into this session</summary>

If you would rather not use Drive, create the folder and upload the files into
it (they will be lost when the session ends):

```python
import os
from google.colab import files
os.makedirs('dataset', exist_ok=True)
%cd dataset
files.upload()   # select auth.txt, proc.txt, flows.txt, dns.txt, redteam.txt
%cd ..
```
</details>

## 5. Run the pipeline

This runs all five steps (build → reason → train → score → evaluate). With a GPU attached
PyKEEN trains and scores on CUDA automatically, and on an A100 the pipeline
enables TF32 tensor cores and uses A100-sized batches.

The defaults are **tuned for a single A100**: they train one strong model
(`RotatE`) with a large `EMBEDDING_DIM` and many `EPOCHS` instead of spreading a
small budget over several models – a narrow, fast run that still gives
significant detection results. Model and speed settings (which models to train,
`EMBEDDING_DIM`, `EPOCHS`, `MAX_TIME`, `EVAL_SAMPLE_SIZE`, `GPU_BATCH_SIZE`, ...)
are constants at the top of `pipeline.py`; edit them there to train on more data,
for longer, or to compare more models now that you have a GPU.

In [ ]:
!python pipeline.py

### Running individual steps

Once the graph is built you can iterate on models without rebuilding it:

```python
!python pipeline.py --steps build
!python pipeline.py --steps train score evaluate
```

## 6. (Optional) Keep the results

Artifacts are written to `generated-files/` and trained models to
`pykeen-lanl-model/<model>/`. Copy them to Drive so they survive the session.

In [ ]:
!mkdir -p /content/drive/MyDrive/kg-results
!cp -r generated-files pykeen-lanl-model /content/drive/MyDrive/kg-results/
print('Copied results to /content/drive/MyDrive/kg-results')